# T11: iLQR for Simple Pendulum

In this tutorial we will implement an iterative Linear-Quadratic Regulator and use it in conjunction with the idea of Receding Horizon Control. We will then use it to achieve a swing-up with the simple pendulum. Lastly, we will add an LQR to the mix to guarantee stability in the goal position.

**Pre-requisites**

- Familiarity with Linear-Quadratic Regulators in general and iterative LQRs in particular.
- Basic understanding of the Receding Horizon concept.

**Goals**

- Implementing an iterative Linear-Quadratic Regulator capable of performing a swing-up with the underactuated simple pendulum.
- Using the Receding Horizon concept to implement a Model Predictive Controller which will drive the system to the goal over the desired horizon when properly set up.

This notebook is organized as follows:

    1. Introduction
    2. Iterative Linear-Quadratic Regulators
    3. iLQR MPC Controller for the simple pendulum
        3.1. Configuring the MPC iLQR controller
        3.2. iLQR Simulation
        3.3. RHC iLQR + LQR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim
import random 
import sympy as smp
import time

%matplotlib inline

from IPython.display import HTML

from pendulum_plant import PendulumPlant, plot_timeseries
from lqr import lqr, dlqr
from utilities_iLQR import check_type
from RoAutils import najafi_based_sampling


## 1. Introduction

First, we set up our PendulumPlant instance:

In [ ]:
mass = 0.06      # mass at the end of the pendulum [kg]
length = 0.1    # length of the rod [m]
damping = 0.0004   # viscious friction [kg m/s]
gravity = 9.81  # gravity [kg m²/s]
inertia = mass*length*length   # inertia of the pendulum [kg m²]
coulomb_fric = 0.0   # Coulomb friction coefficient [-]
torque_limit = np.inf  # torque limit of the motor, here we assume the motor can produce any torque
dt = 0.010            # time step size for iLQR (not simulations) [s]

pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         inertia=inertia,
                         torque_limit=torque_limit)

In order to simulate and animate a motion of the pendulum you can call the simulate_and_animate class method. The arguments are:

    - t0: the initial time
    - y0: the initial state
    - tf: the final time
    - dt: the integration timestep
    - controller: controller object providing torques for the motor (introduced later)
    - integrator: the integration method. options are "euler" and "runge_kutta"
    
The method returns the trajectory split in to time (T), state (X) and torques (U) as well as an animation object.

**Note:** If you do not need the animation you can also call the method *simulate* with the same parameters. The method returns T, X, U but no animation object. Making an animation **takes some time**. 

You can use the plot_timeseries function to plot the trajectories (position, velocity and torque). Of course you can also plot these in any other way you like.

## 2. Iterative Linear Quadratic Regulator

With underactuated systems, sometimes we find that traditional controllers that are limited to reacting to the state of the system are unable to achieve desirable outcomes. One way of implementing a controller which can exploit the dynamics of the system to a greater extent is through the use of trajectory optimization. 

By introducing our knowledge of the system, we can calculate trajectories which can fully exploit the actuator's potential. This also allows us to assign priorities (weights) to different characteristics of the system. We could optimize, for instance, for rapid execution, accuracy, efficiency, etc. Or even balance combinations of priorities.

An iterative Linear Quadratic Regulator (iLQR) achieves this by finding the Linear Quadratic Regulator of a system according to a cost function. Then the system is simulated with the new controller, obtaining a new trajectory. The problem is then linearized around the new trajectory, the LQR algorithm is applied again, and the cycle is repeated successively until convergence.

### 2.1. The iLQR problem

The basic problem statement is as follows:

$$ \min_{u} \; \ell_f (x_N) + \sum^{N-1}_{n=1} \ell (x_n, u_n)$$

$$ x_{n} = f (x_{n-1}, u_{n-1}) \quad \forall n\in[0, N]$$

In other words, we seek to find a set of values of the control input $u$ which minimizes the the cost function. The cost function comprises two distinct terms. $\ell$ is the cost associated to any given step, except for the final step. $\ell$ is a function of the state and control at any given step $k$; $x_n$ and $u_n$, respectively. $\ell_f$ the associated to the final state, which only depends on the state of the system at the final step N.

At the same time, the state of the system at any given instant, $x_n$, other than the initial state $x_0$, is a function of the state and control input at the previous instant. The state from which we start, $x_0$ is supplied by the user.


### 2.2. iLQR calculator class

The first step in our procedure is to define a class which we will call to solve the iLQR problem. Here, we will define the steps we take to obtain the optimal trajectory, which we will later enforce with another controller.

Initially, we define the following member functions:

- **\_\_init\_\_**: Class constructor, initializes symbolic varaibles *x_sym* and *u_sym*.
- **set_start**: Allows us to set an initial state.
- **set_discrete_dynamics**: Allows us to set the dynamics of the system. The dynamics must be already in discrete form, i.e., the inputs are the previous state and control and the output the new state.
- **set_stage_cost**: Allows us to define a cost function for the duration of the motion (except for the final state).
- **set_final_cost**: Allows us to define a cost function for the final state.
- **cost_trj**: Calculates the cost associated to the provided trajectory.

In [ ]:
# The following is adapted from DFKI's iLQR code

# Large parts taken from `Russ Tedrake <https://github.com/RussTedrake/underactuated>`_.

class iLQR_Calculator():
    '''
    Class to calculate an optimal trajectory with an iterative
    linear quadratic regulator (iLQR). This implementation uses sympy.
    '''
    def __init__(self, n_x=2, n_u=1):
        """
        Class to calculate an optimal trajectory with an iterative
        linear quadratic regulator (iLQR). This implementation uses sympy.

        Parameters
        ----------
        n_x : int, default=2
            The size of the state space.
        n_u : int, default=1
            The size of the control space.
        """
        self.n_x = n_x
        self.n_u = n_u

        self.x_sym = smp.symbols("x:"+str(self.n_x))
        self.u_sym = smp.symbols("u:"+str(self.n_u))

    def set_start(self, x0):
        """
        Set the start state for the trajectory.

        Parameters
        ----------
        x0 : array-like
            the start state. Should have the shape of (n_x,)
        """
        self.x0 = x0

    def set_discrete_dynamics(self, dynamics_func):
        '''
        Sets the dynamics function for the iLQR calculation.

        Parameters
        ----------
        danamics_func : function
            dynamics_func should be a function with inputs (x, u) and output xd
        '''
        self.discrete_dynamics = dynamics_func

    def set_stage_cost(self, stage_cost_func):
        '''
        Set the stage cost (running cost) for the ilqr optimization.

        Parameters
        ----------
        stage_cost_func : function
            stage_cost_func should be a function with inputs (x, u)
            and output cost
        '''
        self.stage_cost = stage_cost_func

    def set_final_cost(self, final_cost_func):
        '''
        Set the final cost for the ilqr optimization.

        Parameters
        ----------
        final_cost_func : function
            final_cost_func should be a function with inputs x
            and output cost
        '''
        self.final_cost = final_cost_func

    def cost_trj(self, x_trj, u_trj):
        total = 0.0
        ln = 0.0
        N = x_trj.shape[0]
        for i in range(len(x_trj)-1):
            ln = ln + self.stage_cost(x_trj[i, :], u_trj[i, :]) / N
        lf = self.final_cost(x_trj[i+1, :])
        total = ln + lf
        return total

### 2.3. System dynamics

Our first step is defining the dynamics of our system. In this case, we are dealing with a simple pendulum. We must write the dynamics in the following form:

$$ \dot{x} = A x + B u$$

Where our state variables are position ($\theta$) and velocity ($\dot{\theta}$). The control input is the torque supplied by the actuator.

We define the dynamics in continuous terms. The instant acceleration of the pendulum can be determined using the following equation.

$$ \ddot{\theta} =  \frac{u - m g l \sin \theta - b \dot{\theta}}{I}$$

where $m$ is the mass at the end of the pendulum, $g$ is the acceleration of gravity, $l$ is the length of the pendulum, $b$ is the damping coefficient, and $I$ is the inertia of the pendulum about its base. In this case, we approximate the inertia to $m l^{2}$ using the parallel axis theorem.

In [ ]:
def pendulum_continuous_dynamics(x, u):
    md = check_type(x)

    m = mass
    l = length
    b = damping
    cf = coulomb_fric
    g = gravity
    inertia = m*l*l
    
    pos = x[0]
    vel = x[1]
    if md == smp:
        accn = (u[0] - m * g * l * md.sin(pos) - b * vel -
                cf*md.atan(1e8*vel)*2/np.pi) / inertia
    elif md == np:
        accn = (u[0] - m * g * l * md.sin(pos) - b * vel -
                cf*md.arctan(1e8*vel)*2/np.pi) / inertia
    xd = np.array([vel, accn])
    return xd

To obtain a discrete version of the dynamics, we may apply one of the many explicit integration schemes. Two popular options are the forward Euler scheme and the fourth-order Runge Kutta scheme.

In [ ]:
def pendulum_discrete_dynamics_euler(x, u):
    md = check_type(x)

    x_d = pendulum_continuous_dynamics(x, u)
    x_next = x + x_d*dt
    if md == smp:
        x_next = tuple(x_next)
    return x_next

def pendulum_discrete_dynamics_RK4(x, u):
    md = check_type(x)
    
    k1 = pendulum_continuous_dynamics(x,u)
    k2 = pendulum_continuous_dynamics(x + 0.5*dt*k1,u)
    k3 = pendulum_continuous_dynamics(x + 0.5*dt*k2,u)
    k4 = pendulum_continuous_dynamics(x + dt*k3,u)
    x_d = (k1 + 2 * (k2 + k3) + k4) / 6.0
    x_next = x + x_d*dt
    if md == smp:
        x_next = tuple(x_next)
    return x_next


We add a function to calculate the trajectory from the initial guess of the control input. This function is not used if initial guesses of both the states as well as the control inputs are provided to the calculator.

In [ ]:
def rollout(self, u_trj):
    x_trj = np.zeros((u_trj.shape[0]+1, self.x0.shape[0]))
    x = self.x0
    i = 0
    x_trj[i, :] = x
    for u in u_trj:
        i = i+1
        x = self.discrete_dynamics(x, u)
        x_trj[i, :] = x
    return x_trj

iLQR_Calculator.rollout = rollout

### 2.4. Cost function

Next up, we define the cost function. As stated in section 2.1, the cost function for the iLQR can be chiefly divided in two terms. One represents a penalty for lack of accuracy in the final state. The other is the sum of all penalties during the rest of the process.

Our goal is to get the pendulum to reach its unstable equilibrium point (swing up) within 10 seconds. We also want it to have as low a velocity as possible at the moment it reaches that point. Lastly, we want to minimize the torque given to the pendulum during the motion.

This means that our cost function $J$ is composed of

$$ \ell = C_u u_k^2 $$

$$ \ell_f = C_p (\theta_N - \theta_{goal})^2 + C_v (\dot{\theta}_N - \dot{\theta}_{goal})^2$$

where $C_u$, $C_p$, and $C_v$ are the weights assigned to the control input, position error, and velocity error; respectively.

In [ ]:
def pendulum_swingup_stage_cost(x, u):
    goal=[np.pi, 0]
    Cu=3
    Cp=0.00002
    Cv=0.000004
    Cen=0.0
    m=mass
    l=length
    b=damping
    cf=coulomb_fric
    g=gravity
    
    md = check_type(x)

    eps = 1e-6
    c_pos = (x[0] - goal[0] + eps)**2.0
    c_vel = (x[1] - goal[1] + eps)**2.0
    c_control = u[0]**2.0
    #c_control = u**2.0
    en_g = 0.5*m*(l*goal[1])**2.0 + m*g*l*(1.0-md.cos(goal[0]))
    en = 0.5*m*(l*x[1])**2.0 + m*g*l*(1.0-md.cos(x[0]))
    c_en = (en-en_g+eps)**2.0
    return Cu*c_control + Cp*c_pos + Cv*c_vel + Cen*c_en

def pendulum_swingup_final_cost(x):
    goal=[np.pi, 0]
    Cp=0.1
    Cv=0.02
    Cen=0.0
    m=mass
    l=length
    b=damping
    cf=coulomb_fric
    g=gravity
    
    md = check_type(x)

    eps = 1e-6
    c_pos = (x[0] - goal[0] + eps)**2.0
    c_vel = (x[1] - goal[1] + eps)**2.0
    en_g = 0.5*m*(l*goal[1])**2.0 + m*g*l*(1.0-md.cos(goal[0]))
    en = 0.5*m*(l*x[1])**2.0 + m*g*l*(1.0-md.cos(x[0]))
    c_en = (en-en_g+eps)**2.0
    return Cp*c_pos + Cv*c_vel + Cen*c_en   

More generally, cost functions for an iLQR should be able to be written in linear-quadratic form.

### 2.5. Bellman's Principle of Optimality

Bellman's principle of optimality forms the backbone of the solution to iLQR. It was initially expressed as:

"An optimal policy has the property that whatever the initial state and initial decision are, the remaining decisions must
constitute an optimal policy with regard to the state resulting from the first
decision." (R. Bellman, 1957, p.83)

This can be understood as meaning that we can find an optimal solution to our problem by solving the following equation successively:

$$ V(x_n) = \min_{u_n} \left[ \ell(x_n, u_n) + V(x_{n+1}) \right]$$

where $V$ is the cost to go. Note that at instant $N$, $V$ is only dependant on the state.

We can also define the state and control as:

$$ \bar{x} = x - \delta x$$
$$ \bar{u} = u - \delta u$$

where the nominal trajectory is $\bar{x}$ and $\bar{u}$ and $x$ and $u$ are the new trajectory.

Since the dynamics are linearized around the nominal trajectory, we can decompose the terms of the cost function into the cost incurred as a consequence of the nominal trajectory and that incurred because of the deviations ($\delta x$ and $\delta u$).

Another useful expression, this time for $\ell$, allows us to separate the contributions to the cost term at each instant.

$$\ell(x_n, u_n) = \begin{bmatrix} \ell_{x,n}\\ \ell_{u,n} \end{bmatrix}^T \begin{bmatrix} \delta x_n\\ \delta u_n\end{bmatrix} + \frac{1}{2} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}^T \begin{bmatrix} \ell_{xx, n} & \ell_{ux, n}^T \\ \ell_{ux,n} & \ell_{uu,n}\end{bmatrix} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}$$

We obtain $\ell_{x,n}$ and $\ell_{u,n}$ using the gradient of the cost function at $\bar{x}$,$\bar{u}$. $\ell_{xx,n}$, $\ell_{ux,n}$, and $\ell_{uu,n}$ are partitions of the hessian of the cost functions corresponding to $\bar{x}$ and $\bar{u}$.


In [ ]:
def init_derivatives(self):
    """
    Initialize the derivatives of the dynamics.
    """
    x = self.x_sym
    u = self.u_sym
    
    l = self.stage_cost(x, u)

    l_x = smp.Matrix([l]).jacobian(x)
    self.l_x = smp.lambdify([x, u], l_x, "numpy")

    l_u = smp.Matrix([l]).jacobian(u)
    self.l_u = smp.lambdify([x, u], l_u, "numpy")

    l_xx = smp.Matrix([l_x]).jacobian(x)
    self.l_xx = smp.lambdify([x, u], l_xx, "numpy")

    l_ux = smp.Matrix([l_u]).jacobian(x)
    self.l_ux = smp.lambdify([x, u], l_ux, "numpy")

    l_uu = smp.Matrix([l_u]).jacobian(u)
    self.l_uu = smp.lambdify([x, u], l_uu, "numpy")

    l_final = self.final_cost(x)

    l_final_x = smp.Matrix([l_final]).jacobian(x)
    self.l_final_x = smp.lambdify([x], l_final_x, "numpy")

    l_final_xx = smp.Matrix([l_final_x]).jacobian(x)
    self.l_final_xx = smp.lambdify([x], l_final_xx, "numpy")

    f = self.discrete_dynamics(x, u)
    #print(x)
    f_x = smp.Matrix([f]).jacobian(x)
    self.f_x = smp.lambdify([x, u], f_x, "numpy")

    f_u = smp.Matrix([f]).jacobian(u)
    self.f_u = smp.lambdify([x, u], f_u, "numpy")

def compute_stage_cost_derivatives(self, x, u):
    l_x = np.atleast_1d(np.squeeze(self.l_x(x, u)))
    l_u = np.atleast_1d(np.squeeze(self.l_u(x, u)))
    l_xx = np.atleast_2d(np.squeeze(self.l_xx(x, u)))
    l_ux = np.atleast_2d(np.squeeze(self.l_ux(x, u)))
    l_uu = np.atleast_2d(np.squeeze(self.l_uu(x, u)))
    f_x = np.atleast_2d(np.squeeze(self.f_x(x, u)))
    f_u = np.atleast_2d(np.squeeze(self.f_u(x, u))).T
    return l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u

def compute_final_cost_derivatives(self, x):
    l_final_x = np.atleast_1d(np.squeeze(self.l_final_x(x)))
    l_final_xx = np.atleast_2d(np.squeeze(self.l_final_xx(x)))
    return l_final_x, l_final_xx

iLQR_Calculator.init_derivatives = init_derivatives
iLQR_Calculator.compute_stage_cost_derivatives = compute_stage_cost_derivatives
iLQR_Calculator.compute_final_cost_derivatives = compute_final_cost_derivatives

In the same way, we can also decompose the dynamics function $f$ into two terms. The first is the contribution of the nominal trajectory and the second that of the deviations from the nominal trajectory:

$$ x_{n+1} = f(x_n, u_n) = f(\bar{x}_{n} + \delta x_{n}, \bar{u}_{n} + \delta u_{n}) = f(\bar{x}_n, \bar{u}_n) + \begin{bmatrix} f_{x,n} f_{u,n} \end{bmatrix} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}$$

where $f_{x,n}$ and $f_{u,n}$ are $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial u}$, respectively and evaluated at $\bar{x}$,$\bar{u}$.

If we assume the cost-to-go term to be quadratic, we can do the same:

$$ V(x_{n+1}) = V(\bar{x}_{n+1}) + \delta V(\delta x_{n+1}) = V(\bar{x}_{n+1}) + V_{x,n+1}^T \delta x_{n+1} + \frac{1}{2} \delta x_{n+1}^T V_{xx,n+1} \delta x_{n+1}$$

Now, we can reorganize the problem into a single equation. The cost at each step is then:

$$ Q(x_n, u_n) = Q_n + \begin{bmatrix} Q_{x,n} \\ Q_{u,n} \end{bmatrix}^T \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix} + \frac{1}{2} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}^T \begin{bmatrix} Q_{xx,n} & Q_{ux,n}^T \\ Q_{ux,n} & Q_{uu,n} \end{bmatrix} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}$$

However, our current expression of the cost function is (omitting constant terms):

$$ \delta Q(x_n, u_n) = \begin{bmatrix} \ell_{x,n}\\ \ell_{u,n} \end{bmatrix}^T \begin{bmatrix} \delta x_n\\ \delta u_n\end{bmatrix} + \frac{1}{2} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}^T \begin{bmatrix} \ell_{xx, n} & \ell_{ux, n}^T \\ \ell_{ux,n} & \ell_{uu,n}\end{bmatrix} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix} + V_{x,n+1}^T \delta x_{n+1} + \frac{1}{2} \delta x_{n+1}^T V_{xx,n+1} \delta x_{n+1}$$

which does not fit nicely in with the terms above, which only are only multiplied by $\delta x_n$ and $\delta u_n$. Fortunately, we can use the contribution of the deviations to the dynamics of the system to solve it. From our definition of $f(x_n, u_n)$ it follows that $\delta x_{n+1} = \begin{bmatrix} f_{x,n} & f_{u,n} \end{bmatrix} \begin{bmatrix} \delta x_n \\ \delta u_n \end{bmatrix}$. We can then assemble the new cost function:

$$ \begin{bmatrix} Q_{x,n} \\ Q_{u,n} \end{bmatrix}^T = \begin{bmatrix} \ell_{x,n}\\ \ell_{u,n} \end{bmatrix}^T + V_{x,n+1}^T \begin{bmatrix} f_{x,n} & f_{u,n} \end{bmatrix} $$

$$ \begin{bmatrix} Q_{xx,n} & Q_{ux,n}^T \\ Q_{ux,n} & Q_{uu,n} \end{bmatrix} = \begin{bmatrix} \ell_{xx, n} & \ell_{ux, n}^T \\ \ell_{ux,n} & \ell_{uu,n}\end{bmatrix} + \begin{bmatrix} f_{x,n}^T \\ f_{u,n}^T \end{bmatrix} V_{xx,n+1} \begin{bmatrix} f_{x,n} & f_{u,n} \end{bmatrix}$$

In [ ]:
def Q_terms(self, l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx):
    Q_x = np.matmul(f_x.T, V_x) + l_x
    Q_u = np.matmul(f_u.T, V_x) + l_u
    Q_xx = l_xx + np.matmul(f_x.T, np.matmul(V_xx, f_x))
    Q_ux = l_ux + np.matmul(f_u.T, np.matmul(V_xx, f_x))
    Q_uu = l_uu + np.matmul(f_u.T, np.matmul(V_xx, f_u))
    return Q_x, Q_u, Q_xx, Q_ux, Q_uu

iLQR_Calculator.Q_terms = Q_terms

We can now derive the optimal controller for each time step by minimizing $\delta V$. 

$$ \delta u^* = \underset{\delta u}{\operatorname{argmin}} \delta Q $$

$$ \frac{\partial \delta Q}{\partial \delta u} = Q_u + Q_{uu} \delta u + Q_{ux} \delta x$$

$$ \delta u^* = K \delta x + k = - Q_{uu}^{-1} Q_{ux} \delta x - Q_{uu}^{-1} Q_u$$

To use this definition of the optimal controller to calculate $ V(x_n) $ from $Q$, just substitute $\delta u$ with the control law. The reduction in cost we can expect is the value difference between the value we obtained for $Q$ and $Q$ if $\delta x$ and $\delta u$ are zero. The more accurate our linearization, the closer our reduction will be to the estimate.

In [ ]:
def gains(self, Q_uu, Q_u, Q_ux):
    #Q_uu_inv = np.linalg.inv(Q_uu + Q_uu.T) # why the addition of Quu transp.??
    Q_uu_inv = np.linalg.inv(Q_uu)
    #k = -2*np.matmul(Q_uu_inv, Q_u)
    #K = -2*np.matmul(Q_uu_inv, Q_ux) #multiplier?
    k = -np.matmul(Q_uu_inv, Q_u)
    K = -np.matmul(Q_uu_inv, Q_ux) #multiplier?

    k = np.atleast_1d(np.squeeze(k))
    K = np.atleast_2d(np.squeeze(K))
    return k, K

def V_terms(self, Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k):
    V_x = Q_x - np.matmul(K.T, np.matmul(Q_uu, k))
    V_xx = Q_xx - np.matmul(K.T, np.matmul(Q_uu, K))
    return V_x, V_xx

def expected_cost_reduction(self, Q_u, Q_uu, k):
    ecr = -Q_u.T*k - 0.5*k.T*(Q_uu*k)
    return np.squeeze(ecr)

iLQR_Calculator.gains = gains
iLQR_Calculator.V_terms = V_terms
iLQR_Calculator.expected_cost_reduction = expected_cost_reduction

### 2.6. Forward pass

As we stated earlier, iLQR is divided between forward and backward passes. The role of the forward pass is to calculate the nominal trajectory. From $x_0$ to $x_N$ we simply apply the dynamics taking into account the contribution of the control using the control law. Note that, when applying the dynamics equation, we should use the complete expression of the control law, which includes the input from the previous nominal trajectory:

$$ u_{n, new} = \bar{u}_n + k_n + K_n(x_{n,new} - \bar{x}_n)$$

In [ ]:
def forward_pass(self, x_trj, u_trj, k_trj, K_trj):
    x_trj_new = np.zeros(x_trj.shape)
    x_trj_new[0, :] = x_trj[0, :]
    u_trj_new = np.zeros(u_trj.shape)
    for n in range(u_trj.shape[0]):
        u_trj_new[n, :] = u_trj[n] + k_trj[n] \
                          + np.matmul(K_trj[n],
                                      ((x_trj_new[n] - x_trj[n])))
        x_trj_new[n+1, :] = self.discrete_dynamics(x_trj_new[n],
                                                   u_trj_new[n])
    return x_trj_new, u_trj_new

iLQR_Calculator.forward_pass = forward_pass

### 2.7. Backward pass

In the backward pass we use the new nominal trajectory we generated. We start from instant $N$ and make our way back to the initial state. For each instant, we compute the terms of $\ell$, then those of $Q$, $K$ and $k$.

To ensure that numeric imprecision does not make $Q_{uu}$ non-invertible, we include a regularization step for that term. This is implemented as a diagonal matrix which is added to $Q_{uu}$. A side effect of this addition is that straying from the nominal trajectory $\bar{u}$ is penalized, which makes step size smaller.

In [ ]:
def backward_pass(self, x_trj, u_trj, regu):
    k_trj = np.zeros([u_trj.shape[0], u_trj.shape[1]])
    K_trj = np.zeros([u_trj.shape[0], u_trj.shape[1], x_trj.shape[1]])
    expected_cost_redu = 0
    l_final_x, l_final_xx = self.compute_final_cost_derivatives(x_trj[-1])
    V_x = l_final_x
    V_xx = l_final_xx
    for n in range(u_trj.shape[0]-1, -1, -1):
        l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u = self.compute_stage_cost_derivatives(x_trj[n], u_trj[n])
        Q_x, Q_u, Q_xx, Q_ux, Q_uu = self.Q_terms(l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx)
        # We add regularization to ensure that Q_uu is invertible and nicely conditioned
        Q_uu_regu = Q_uu + np.eye(Q_uu.shape[0])*regu
        k, K = self.gains(Q_uu_regu, Q_u, Q_ux)
        k_trj[n, :] = k
        K_trj[n, :, :] = K
        V_x, V_xx = self.V_terms(Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k)
        expected_cost_redu += self.expected_cost_reduction(Q_u, Q_uu, k)
    return k_trj, K_trj, expected_cost_redu

iLQR_Calculator.backward_pass = backward_pass

### 2.8. Main loop

Finally, we define avalue at which we will stop if we don't expect much improvement and a maximum amount of iterations in case we fail to converge.

The structure of algorithm is as follows:

- Run **rollout** to obtain the first trajectory.
- Main loop:
    - Call **backward_pass** to generate the new control law.
    - Call **forward_pass** to generate the new trajectory.
    - Repeat until convergence or reaching maximum iterations. 

Every time we run the forward pass, if the new trajectory has a higher cost associated than the previous one, we reject it and run it again with higher regularization. If the iteration is successful in finding a better trajectory, we decrease the regularization.

In [ ]:
def run_ilqr(self, N=50, init_u_trj=None, init_x_trj=None, shift=False,
             max_iter=50, break_cost_redu=1e-6, regu_init=100):
    """
    Run the iLQR optimization and receive a optimal trajectory for the
    defined cost function.

    Parameters
    ----------
    N : int, default=50
        The number of waypoints for the trajectory
    init_u_trj : array-like, default=None
        initial guess for the control trajectory
        ignored if None
    init_x_trj : array_like, default=None
        initial guess for the state space trajectory
        ignored if None
    shift : bool, default=False
        whether to shift the initial guess trajectories by one entry
        (delete the first entry and duplicate the last entry)
    max_iter : int, default=50
        optimization iterations the alogrithm makes at every timestep
    break_cost_redu : float, default=1e-6
        cost at which the optimization breaks off early
    regu_init : float, default=100
       initialization value for the regularizer

    Returns
    -------
    x_trj : array-like
        state space trajectory
    u_trj : array-like
        control trajectory
    cost_trace : array-like
        trace of the cost development during the optimization
    regu_trace : array-like
        trace of the regularizer development during the optimization
    redu_ratio_trace : array-like
        trace of ratio of cost_reduction and expected cost reduction
         during the optimization
    redu_trace : array-like
        trace of the cost reduction development during the optimization
    """
    # First forward rollout
    
    if init_u_trj is not None:
        u_trj = init_u_trj
        if shift:
            u_trj = np.delete(u_trj, 0, axis=0)
            u_trj = np.append(u_trj, [u_trj[-1]], axis=0)
    else:
        u_trj = np.random.randn(N-1, self.n_u)*0.0001

    if init_x_trj is not None:
        x_trj = init_x_trj
        if shift:
            x_trj = np.delete(x_trj, 0, axis=0)
            x_trj[0,:] = self.x0
            x_trj = np.append(x_trj,
                              [self.discrete_dynamics(x_trj[-1],
                               np.array(u_trj[-1]))], axis=0)
    else:
        x_trj = self.rollout(u_trj)

    total_cost = self.cost_trj(x_trj, u_trj)
    regu = regu_init
    max_regu = 10000
    min_regu = 0.01

    # Setup traces
    cost_trace = [total_cost]
    redu_ratio_trace = [1]
    redu_trace = []
    regu_trace = [regu]

    # Run main loop
    for it in range(max_iter):
        # Backward and forward pass
        k_trj, K_trj, expected_cost_redu = self.backward_pass(x_trj,
                                                              u_trj,
                                                              regu)
        x_trj_new, u_trj_new = self.forward_pass(x_trj, u_trj,
                                                 k_trj, K_trj)
        # Evaluate new trajectory
        total_cost = self.cost_trj(x_trj_new, u_trj_new)
        cost_redu = cost_trace[-1] - total_cost
        redu_ratio = cost_redu / abs(expected_cost_redu)
        # Accept or reject iteration
        if cost_redu > 0:
            # Improvement! Accept new trajectories and lower regularization
            redu_ratio_trace.append(redu_ratio)
            cost_trace.append(total_cost)
            x_trj = x_trj_new
            u_trj = u_trj_new
            regu *= 0.7
        else:
            # Reject new trajectories and increase regularization
            regu *= 2.0
            cost_trace.append(cost_trace[-1])
            redu_ratio_trace.append(0)
            if it > 10:
                print('over')
        regu = min(max(regu, min_regu), max_regu)
        regu_trace.append(regu)
        redu_trace.append(cost_redu)

        # Early termination if expected improvement is small
        if expected_cost_redu <= break_cost_redu or (cost_trace[-2]-cost_trace[-1]) < 1e-6:
            break

    return x_trj, u_trj, cost_trace, regu_trace, redu_ratio_trace, redu_trace, k_trj, K_trj

iLQR_Calculator.run_ilqr = run_ilqr

### 2.9. iLQR Controller for a simple pendulum

So far, we talked about using iLQR for trajectory optimization. If we want to use it on a real system, we will need to somehow enforce that trajectory. For the actual controller which will do that, we define a new class, **iLQRController**. From **iLQRController** we will run the optimization and then enforce it using a PD controller. For the PD controller, we use the Riccati gains.

In [ ]:
class iLQRController():
    def __init__(self, mass, length, damping, torque_limit, gravity, n_x = 2, n_u = 1, dt = 0.01):
        self.m = mass
        self.l = length
        self.g = gravity
        self.b = damping
        self.torque_limit = torque_limit
        self.i = 0
        self.iLQR = iLQR_Calculator(n_x = n_x, n_u = n_u)

    def set_discrete_dynamics(self,dyn_func):
        self.iLQR.set_discrete_dynamics(dyn_func)

    def set_stage_cost(self,s_cost):
        self.iLQR.set_stage_cost(s_cost)

    def set_final_cost(self,f_cost):
        self.iLQR.set_final_cost(f_cost)

    # def init_derivatives(self):
    #     self.iLQR.init_derivatives()

    def set_start(self, x0):
        self.iLQR.set_start(x0)

    def run_ilqr(self, N, init_u_trj, init_x_trj, max_iter, regu_init, break_cost_redu=1e-6):
        self.iLQR.init_derivatives()
        self.x_trj, self.u_trj, self.cost_trace, self.regu_trace, self.redu_ratio_trace, self.redu_trace, self.k_trj, self.K_trj = self.iLQR.run_ilqr(N, init_u_trj, init_x_trj, max_iter, regu_init, break_cost_redu)
        self.N = N
    
    def restart_trajectory(self):
        self.i = 0
        
    def get_control_output(self,x):
        # PD controller is used to enforce the trajectory calculated by the iLQR
        
        if self.i>self.N-1.5:
            #print("REACHED END OF TRAJECTORY")
            tau = 0.0
        else:

            u = self.u_trj[self.i][0] + self.k_trj[self.i] \
                          + np.matmul(self.K_trj[self.i],
                                      ((x - self.x_trj[self.i])))
            u = u.item(0)
            tau = np.clip(u, -self.torque_limit, self.torque_limit)
            self.i += 1

        return tau

### 2.10. Configuring the iLQR controller

When seting up the iLQR controller, we must create an instance of the class **iLQRController** with the dynamic properties of the pendulum, and the maximum torque we will allow (symmetric).

Then we add the dynamics equation and cost functions. We also set the initial state. Lastly we run the ilqr offline. This means that we are optimizing the trajectory before starting the motion.

In [ ]:
ilqr_controller = iLQRController(mass=mass, length=length, damping = damping, torque_limit=0.03, gravity=gravity)
ilqr_controller.set_discrete_dynamics(pendulum_discrete_dynamics_RK4)
ilqr_controller.set_stage_cost(pendulum_swingup_stage_cost)
ilqr_controller.set_final_cost(pendulum_swingup_final_cost)

x0 = np.array([0.001, 0.0])
ilqr_controller.set_start(x0)
ilqr_controller.run_ilqr(N = 1000, init_u_trj=None, init_x_trj=None, max_iter=100, regu_init=100, break_cost_redu=1e-6)

### 2.11. Simulation

To check that our iLQR controller works as intended, we can simulate its output. Since the trajectory optimization is done offline and enforced by a PD controller, if we plan to execute the motion again, we will need to call **iLQRController.restart_trajectory()** to start again.

### Think-Pair-Share (10 min)

Go back to section 2.4. and set the correct weights in functions *pendulum_swingup_stage_cost* and *pendulum_swingup_final_cost* to achieve a swing-up with as little torque as possible. Test the solution to make sure the controller achieves the maneuver.

In [ ]:
%%capture
T, X, U, anim = pendulum.simulate_and_animate(
              t0=0.0,
              y0=[0.001, 0.0], #y0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=ilqr_controller,
              integrator="runge_kutta")
ilqr_controller.restart_trajectory()

In [ ]:
HTML(anim.to_html5_video())

In [ ]:
plot_timeseries(T, X, U)

In [ ]:
Tsim_es, Xsim_es, Usim_es = pendulum.simulate(
              t0=0.0,
              y0=[0.001, 0],
              tf=10.0,
              dt=0.01,
              controller=ilqr_controller,
              integrator="runge_kutta")
plot_timeseries(Tsim_es, Xsim_es, Usim_es)


# Export the data to csv file
simulation_csv_data_es = np.array([Tsim_es, np.asarray(Xsim_es).T[0], np.asarray(Xsim_es).T[1],Usim_es]).T
np.savetxt("es_simulation_data_es.csv", simulation_csv_data_es, delimiter=',', header="time,pos,vel,tau", comments="")
ilqr_controller.restart_trajectory()

In [ ]:
plt.plot(Tsim_es[1:-1], ilqr_controller.u_trj, label="u")
plt.ylabel("Torque [Nm]")
plt.xlabel("Time [s]")
plt.legend(loc="best")
plt.show()

## 3. iLQR MPC Controller for the simple pendulum

The previous version of the iLQR calculates the trajectory once and then will try to follow it. This can make it hard for the controller to respond to disturbances in an effective way. For the actual controller which will do that, we define a new class, **iLQR_MPC_Controller**. From **iLQR_MPC_Controller** we will run the optimization periodically and then have the motor output the first torque command of the calculated trajectory. As you know, this technique is known by the name of Receding Horizon Control.

In [ ]:
class iLQR_MPC_Controller():
    import pyCandle
    import time

    def __init__(self, mass, length, damping, torque_limit, gravity, Kp, Kd, N, 
                 max_iter=100, regu_init=100, break_cost_redu=5e-1, 
                 init_u_trj=None, init_x_trj=None, n_x=2, n_u=1, dt=0.02):
        self.Kp = Kp
        self.Kd = Kd
        self.m = mass
        self.l = length
        self.g = gravity
        self.b = damping
        self.torque_limit = torque_limit
        self.N = N
        self.dt = dt
        self.i = 0
        self.iLQR = iLQR_Calculator(n_x = n_x, n_u = n_u)
        self.max_iter = max_iter
        self.break_cost_redu = break_cost_redu
        self.regu_init = regu_init
        self.init_u_trj = init_u_trj
        self.init_x_trj = init_x_trj
        self.x_trj_archive = np.array([]).reshape(0,self.N)
        self.u_trj_archive = np.array([]).reshape(0,self.N-1)
        self.cost_archive = []
        self.ite_archive = []

    def set_discrete_dynamics(self,dyn_func):
        self.iLQR.set_discrete_dynamics(dyn_func)

    def set_stage_cost(self,s_cost):
        self.iLQR.set_stage_cost(s_cost)

    def set_final_cost(self,f_cost):
        self.iLQR.set_final_cost(f_cost)

    def set_start(self, x0):
        self.iLQR.set_start(x0)

    def run_ilqr(self, N, init_u_trj, init_x_trj):
        self.iLQR.init_derivatives()
        self.x_trj, self.u_trj, self.cost_trace, self.regu_trace, self.redu_ratio_trace, self.redu_trace = self.iLQR.run_ilqr(self.N, self.init_u_trj, self.init_x_trj, self.max_iter, self.regu_init, self.break_cost_redu)
    
    def init_derivatives(self):
        self.iLQR.init_derivatives()
    
    def restart_trajectory(self):
        self.i = 0

    def set_current_state(self,x):
        self.xc = x

    def get_trajectory_history(self):
        return self.x_trj_archive

    def get_control_history(self):
        return self.u_trj_archive

    def get_cost_history(self):
        return self.cost_archive
        
    def get_ite_history(self):
        return self.ite_archive
        
    def get_animation_data(self):
        return self.get_trajectory_history(), self.get_control_history(), self.dt, self.N

    def get_control_output(self,x):
        #print(x)
        
        self.iLQR.set_start(np.array(x))
        self.x_trj, self.u_trj, self.cost_trace, self.regu_trace, self.redu_ratio_trace, self.redu_trace = self.iLQR.run_ilqr(self.N, self.init_u_trj, None, True, self.max_iter, self.break_cost_redu, self.regu_init)[:6]
        tau = self.u_trj[0].item()
        self.init_u_trj = self.u_trj
        self.init_x_trj = self.x_trj
        
        self.x_trj_archive = np.vstack((self.x_trj_archive, np.asarray(self.x_trj).T))
        self.u_trj_archive = np.vstack((self.u_trj_archive, np.asarray(self.u_trj).T))
        self.cost_archive.append(self.cost_trace[0])
        self.ite_archive.append(len(self.cost_trace))
        
        return np.clip(tau, -self.torque_limit, self.torque_limit)

### 3.1. Configuring the MPC iLQR controller

When seting up the iLQR controller, we must create an instance of the class **iLQR_MPC_Controller** with the dynamic properties of the pendulum, the maximum torque we will allow (symmetric), the control horizon, and settings for the optimization.

Then we add the dynamics equation and cost functions. We also set the initial state. For the MPC version of our iLQR we will use the following cost functions:

In [ ]:
def pendulum_swingup_stage_cost_norm(x, u):
    goal=[np.pi, 0]
    #Cu= 0.6
    #Cp= 0.5
    #Cv= 0.2
    #Cen= 0.0
    #Cu= 1.5
    #Cp= 0.05
    #Cv= 0.025
    Cu= 1.0
    Cp= 0.5
    Cv= 0.325
    Cen= 0.1
    m=mass
    l=length
    b=damping
    cf=coulomb_fric
    g=gravity
    
    md = check_type(x)

    eps = 1e-6
    c_pos = ((x[0] - goal[0])/(2*goal[0]))**2.0 + eps
    c_vel = ((x[1] - goal[1])/15)**2.0 + eps
    c_control = (u[0]/0.03)**2.0 + eps
    en_g = 0.5*m*(l*goal[1])**2.0 + m*g*l*(1.0-md.cos(goal[0]))
    en = 0.5*m*(l*x[1])**2.0 + m*g*l*(1.0-md.cos(x[0]))
    c_en = ((en-en_g)/(m*g*l*(1.0-md.cos(np.pi))))**2.0 + eps
    return Cu*c_control + Cp*c_pos + Cv*c_vel + Cen*c_en

def pendulum_swingup_final_cost_norm(x):
    goal=[np.pi, 0]
    #Cp= 5.0
    #Cv= 2.0
    #Cen= 30.0
    Cp= 1.0
    Cv= 0.5
    Cen= 30.0
    m=mass
    l=length
    b=damping
    cf=coulomb_fric
    g=gravity
    
    md = check_type(x)

    eps = 1e-6
    c_pos = ((x[0] - goal[0])/(2*goal[0]))**2.0 + eps
    c_vel = ((x[1] - goal[1])/15)**2.0 + eps
    en_g = 0.5*m*(l*goal[1])**2.0 + m*g*l*(1.0-md.cos(goal[0]))
    en = 0.5*m*(l*x[1])**2.0 + m*g*l*(1.0-md.cos(x[0]))
    c_en = ((en-en_g)/(m*g*l*(1.0-md.cos(np.pi))))**2.0 + eps
    return Cp*c_pos + Cv*c_vel + Cen*c_en

### Think-Pair-Share (10 min)

Set the correct weights in functions *pendulum_swingup_stage_cost_norm* and *pendulum_swingup_final_cost_norm* to achieve a swing-up with as little torque as possible. Test the solution to make sure the MPC controller achieves the maneuver.

In [ ]:
ilqr_MPC_controller = iLQR_MPC_Controller(mass=mass, length=length, damping = damping, torque_limit=0.03, gravity=gravity, Kp=0.1, Kd = 0.01, N=50, max_iter=50, regu_init=100, break_cost_redu=5e-1, dt = 0.01)
ilqr_MPC_controller.set_discrete_dynamics(pendulum_discrete_dynamics_RK4)
ilqr_MPC_controller.set_stage_cost(pendulum_swingup_stage_cost_norm)
ilqr_MPC_controller.set_final_cost(pendulum_swingup_final_cost_norm)
ilqr_MPC_controller.init_derivatives()

x0 = np.array([0.001, 0.0])
ilqr_MPC_controller.set_start(x0)

### 3.2. iLQR Simulation

To check that our iLQR controller works as intended, we can simulate its output.

In [ ]:
%%capture
T, X, U, anim = pendulum.simulate_and_animate(
              t0=0.0,
              y0=[0.001, 0.0], #y0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=ilqr_MPC_controller,
              integrator="runge_kutta")
ilqr_controller.restart_trajectory()

In [ ]:
HTML(anim.to_html5_video())

In [ ]:
plot_timeseries(T, X, U)

In [ ]:
t1 = time.time()
Tsim_es, Xsim_es, Usim_es = pendulum.simulate(
              t0=0.0,
              y0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=ilqr_MPC_controller,
              integrator="runge_kutta")
plot_timeseries(Tsim_es, Xsim_es, Usim_es)
t2 = time.time()
print(t2-t1)

# Export the data to csv file
simulation_csv_data_es = np.array([Tsim_es, np.asarray(Xsim_es).T[0], np.asarray(Xsim_es).T[1],Usim_es]).T
np.savetxt("es_simulation_data_es.csv", simulation_csv_data_es, delimiter=',', header="time,pos,vel,tau", comments="")
ilqr_MPC_controller.restart_trajectory()

In [ ]:
plt.plot(Tsim_es, np.asarray(Xsim_es).T[0], label="theta")
plt.plot(Tsim_es, Usim_es, label="u")
plt.show()

We can also visualize the trajectories that the controller generates everytime it provides a torque output. Run the cell below to generate an animation that shows some of the trajectories calculated during the previous simulation, as well as to plot the controller's output.

In [ ]:
pendulum.animate_MPC_plot2(ilqr_MPC_controller.get_trajectory_history(), np.asarray(ilqr_MPC_controller.get_control_history()), ilqr_MPC_controller.get_cost_history(), ilqr_MPC_controller.get_ite_history(), 0.01, 50, vid = True, vidname = "video_air1")

plt.plot(Tsim_es, Usim_es, label="u")
plt.xlabel("Time [s]")
plt.ylabel("Torque [Nm]")
plt.legend(loc="best")
plt.show()

### 3.3. RHC iLQR + LQR

As we can see from our testing, the iLQR trajectory optimization may not be quick enough to stop the pendulum from falling when there is a disturbance. We can use the concept of Region of Attraction that we learned about in the previous tutorial to stabilize the pendulum in the upright position.

In [ ]:
class LQRController():
    def __init__(self, mass, length, gravity, damping, torque_limit, Q, R):
        self.Q = Q                 # 2x2 Matrix cost on state error
        self.R = R                 # 1x1 Matrix cost on actuation
        self.m = mass
        self.l = length
        self.g = gravity
        self.b = damping
        self.torque_limit = torque_limit
        A, B = self.linearize()
        self.K = np.zeros((2, 2))
        # Continuous Time LQR
        self.K, self.S, _ = lqr(A, B, Q, R)
        print("LQR Gain Matrix (K):", self.K)
        # Discrete Time LQR
        # self.K, _, _ = dlqr(A, B, Q, R)
    
    def linearize(self):
        # Linearized equations of motion for simple pendulum
        # at the upright/topmost state (pi, 0)
        # Either Continuous form: x_dot = Ax + Bu
        # Or discrete form: x[n+1] = Ax[n] + Bu[n]
        # This function returns the linear A and B matrices
        A = np.array([[0, 1], [self.g/self.l , -self.b/(self.m*(self.l)**2)]])
        B = np.array([[0],[1 / (self.m * (self.l)**2)]])

        return A, B

    def get_control_output(self, x):
        # Use the LQR Gain matrix K to create the controller.
        tau = -self.K.dot(x - np.array([np.pi, 0]))
        # Add a torque limit before returning the tau value.
        return np.clip(tau[0], -self.torque_limit, self.torque_limit)

In [ ]:
class iLQRAndLQRController():
    """
    Controller which swings up the pendulum with the energy shaping
    controller and stabilizes the pendulum with the lqr controller.
    """
    def __init__(self, mass=1.0, length=0.5, damping=0.1,
                 gravity=9.81, torque_limit=0.03, N=50, K=1.0,
                 Q=np.diag((10, 1)), R=np.array([[1]]), compute_RoA=False):
        self.m = mass
        self.l = length
        self.b = damping
        self.g = gravity
        self.mode = 0
        
        self.ilqr_controller = iLQR_MPC_Controller(mass=mass, 
                                                   length=length, 
                                                   damping = damping, 
                                                   torque_limit=0.03, 
                                                   gravity=gravity, 
                                                   Kp=0.1, 
                                                   Kd = 0.01, 
                                                   N=50, 
                                                   max_iter=50, 
                                                   regu_init=100, 
                                                   break_cost_redu=5e-1, 
                                                   dt = 0.01)
        
        self.lqr_controller = LQRController(mass=mass,
                                            length=length,
                                            damping=damping,
                                            gravity=gravity,
                                            torque_limit=torque_limit,
                                            Q=Q,
                                            R=R)

        self.active_controller = "none"
        self.swingup_time = 10.0
        dt = 0.01
        self.eps = [0.2, 10.0]

    def set_RoA(self, M, rho):
        self.M = M
        self.rho = rho

    def is_in_RoA(self, x):

        x = np.array(x)

        if (x.T @ self.M @ x) < self.rho:
            return True
        else:
            return False

    def get_control_output(self, x):
        # Measured State
        meas_pos = x[0] 
        meas_vel = x[1]
        # Desired Goal State
        des_pos = np.pi 
        des_vel = 0.0
        

        u = self.ilqr_controller.get_control_output(x)
        th = meas_pos + np.pi
        th = (th + np.pi) % (2*np.pi) - np.pi
        
        if self.is_in_RoA([th, meas_vel]):
            if u is not None:
                if self.active_controller != "lqr":
                    self.active_controller = "lqr"
                    print("Switching to lqr")
                u = self.lqr_controller.get_control_output(x)
            else:
                if self.active_controller != "iLQR":
                    self.active_controller = "iLQR"
                    print("Switching to iLQR")
                u = self.ilqr_controller.get_control_output(x)
        return u

In [ ]:
Q = np.array([[0.001, 0], [0, 0.0001]])
R = np.array([[1]])
K = 0.1
ilqr_lqr_controller = iLQRAndLQRController(mass=mass, length=length, damping=damping, gravity=gravity, torque_limit=0.03, K=K, Q=Q, R=R)
rho, M, points = najafi_based_sampling(pendulum, ilqr_lqr_controller.lqr_controller, n=1000)
ilqr_lqr_controller.set_RoA(M, rho)

In [ ]:
ilqr_lqr_controller.ilqr_controller.set_discrete_dynamics(pendulum_discrete_dynamics_RK4)
ilqr_lqr_controller.ilqr_controller.set_stage_cost(pendulum_swingup_stage_cost_norm)
ilqr_lqr_controller.ilqr_controller.set_final_cost(pendulum_swingup_final_cost_norm)
ilqr_lqr_controller.ilqr_controller.init_derivatives()
ilqr_lqr_controller.ilqr_controller.set_start(x0)

Simulate the controller to make sure it works correctly before running it on the real hardware.

In [ ]:
Tsim_es_lqr, Xsim_es_lqr, Usim_es_lqr = pendulum.simulate(
              t0=0.0,
              y0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=ilqr_lqr_controller,
              integrator="runge_kutta")
plot_timeseries(Tsim_es_lqr, Xsim_es_lqr, Usim_es_lqr)

# Measured Position
plt.figure
plt.plot(Tsim_es_lqr, np.asarray(Xsim_es_lqr).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()
# Measured Velocity
plt.figure
plt.plot(Tsim_es_lqr, np.asarray(Xsim_es_lqr).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()
# Measured Torque
plt.figure
plt.plot(Tsim_es_lqr, Usim_es_lqr)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
simulation_csv_data_es_lqr = np.array([Tsim_es_lqr, np.asarray(Xsim_es_lqr).T[0], np.asarray(Xsim_es_lqr).T[1],Usim_es_lqr]).T
np.savetxt("es_simulation_data_es_lqr.csv", simulation_csv_data_es_lqr, delimiter=',', header="time,pos,vel,tau", comments="")

## 4. Hardware Experiments with the real pendulum

The next cells apply the controllers on the real pendulum. If the notebook runs on the CloudPendulum server, the experiments are run on the cloud (your user token is read from `~/.env`). If it runs on a computer that is physically connected to a pendulum, the experiments are run locally and no token is needed.

In [ ]:
try:
    from pycloudpendulum import Client
    USE_LOCAL = False
except:
    from cloud_pendulum_local import Client
    USE_LOCAL = True

import os
from IPython.display import Video, display

# the token is read from ~/.env on the cloud; no token is needed locally
if USE_LOCAL:
    user_token = "dummy_token"
else:
    with open(os.path.expanduser("~/.env"), 'r') as file:
        user_token = file.read().strip('\n')

The following image shows a hardware prototype of an active pendulum system actuated with a small direct drive actuator (GL35 from T-motors). The control electronics are provided by MAB robotics. In future, this system will be integrated in Chalmers Cloud infrastructure and students will be able to remotely access the pendulum and get live experimental data from it with the comfort of their web-browsers. You get to be alpha testers of the system!

<center><img src="media/pendulum_hardware.jpeg" width="400"></center>

### 4.1. Offline iLQR experiment

In [ ]:
tf = 20 # Final time (s)
dt = 0.01 # Time step (s)
Treal_es, Xreal_es, Ureal_es, Ureal_es_des, video_url_es, video_path_es = pendulum.run_on_hardware(user_token=user_token, x0=0.0, tf=tf, dt=dt, controller=ilqr_controller, save_video=not USE_LOCAL)

# Measured Position
plt.figure
plt.plot(Treal_es, np.asarray(Xreal_es).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_es, np.asarray(Xreal_es).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_es, Ureal_es)
plt.plot(Treal_es, Ureal_es_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
measured_csv_data_es = np.array([Treal_es, np.asarray(Xreal_es).T[0], np.asarray(Xreal_es).T[1],Ureal_es]).T
np.savetxt("es_measured_data_es.csv", measured_csv_data_es, delimiter=',', header="time,pos,vel,tau", comments="")

# Show the video (only available on the cloud)
if not USE_LOCAL:
    display(Video(video_path_es, embed=True))

### 4.2. iLQR MPC Experiment

In [ ]:
tf = 10 # Final time (s)
dt = 0.01 # Time step (s)
Treal_lqr, Xreal_lqr, Ureal_lqr, Ureal_lqr_des, video_url_lqr, video_path_lqr = pendulum.run_on_hardware(user_token=user_token, x0=0.0, tf=tf, dt=dt, controller=ilqr_MPC_controller, save_video=not USE_LOCAL)

# Measured Position
plt.figure
plt.plot(Treal_lqr, np.asarray(Xreal_lqr).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_lqr, np.asarray(Xreal_lqr).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_lqr, Ureal_lqr)
plt.plot(Treal_lqr, Ureal_lqr_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
measured_csv_data_lqr = np.array([Treal_lqr, np.asarray(Xreal_lqr).T[0], np.asarray(Xreal_lqr).T[1],Ureal_lqr]).T
np.savetxt("es_measured_data_lqr.csv", measured_csv_data_lqr, delimiter=',', header="time,pos,vel,tau", comments="")

# Show the video (only available on the cloud)
if not USE_LOCAL:
    display(Video(video_path_lqr, embed=True))

### 4.3. iLQR MPC + LQR Experiment

In [ ]:
#ilqr_lqr_controller.ilqr_controller.restart_trajectory()
tf = 10 # Final time (s)
dt = 0.01 # Time step (s)
Treal_es_lqr, Xreal_es_lqr, Ureal_es_lqr, Ureal_es_lqr_des, video_url_es_lqr, video_path_es_lqr = pendulum.run_on_hardware(user_token=user_token, x0=0.0, tf=tf, dt=dt, controller=ilqr_lqr_controller, save_video=not USE_LOCAL)

# Measured Position
plt.figure
plt.plot(Treal_es_lqr, np.asarray(Xreal_es_lqr).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_es_lqr, np.asarray(Xreal_es_lqr).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_es_lqr, Ureal_es_lqr)
plt.plot(Treal_es_lqr, Ureal_es_lqr_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
measured_csv_data_es_lqr = np.array([Treal_es_lqr, np.asarray(Xreal_es_lqr).T[0], np.asarray(Xreal_es_lqr).T[1],Ureal_es_lqr]).T
np.savetxt("es_measured_data_es_lqr.csv", measured_csv_data_es_lqr, delimiter=',', header="time,pos,vel,tau", comments="")

# Show the video (only available on the cloud)
if not USE_LOCAL:
    display(Video(video_path_es_lqr, embed=True))